# Cosmos3 · VANTAGE evaluation

**Prepare → connect → five real predictions → full submission**

This notebook is a thin front end to `launchable.py`, using the official public VANTAGE harness by default. The full eight-task run is **opt-in**: **Run All runs the five-question smoke only** while `RUN_FULL = False`.

**Before opening:** run `bash setup.sh` on the Brev VM from the launchable directory. Use the authenticated Brev link to port 8888. Docker setup/lifecycle commands run on the VM host, not here; this evaluator has no Docker socket. See [README](README.md) for launch fields and credentials.

**Validation boundary:** GPU serving, runtime, end-to-end full evaluation, portal acceptance, and reference-score parity are not yet GPU-verified. Five predictions are not a leaderboard score.

## 1 · Configure a fresh run

The pinned Nano model and public VQA data support anonymous access; no HF token or hidden prompt is needed for this smoke. An inherited secret `HF_TOKEN` remains available for gated assets. Full SOT preparation requires a read token with accepted [PhysicalAI-SmartSpaces access terms](https://huggingface.co/datasets/nvidia/PhysicalAI-SmartSpaces). Public mode does **not** need internal AWS/NGC/dataset-service or API judge keys. Existing endpoint/model environment settings are preserved.

Use a new timestamped directory for each run. Re-running this cell creates new output paths; do not mix smoke and full outputs. Change `SOURCE` to `internal` only after configuring the host AWS mount described at the end.

In [ ]:
import os
import subprocess
import sys
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from IPython.display import FileLink, display

SOURCE = 'public'
RUN_FULL = False  # Explicitly opt in before running section 6.
LAUNCHABLE = Path('/opt/vlmevalkit/launchable/launchable.py')
CLI = [sys.executable, str(LAUNCHABLE)]
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SMOKE_WORK = Path('/results') / f'{SOURCE}-smoke-{STAMP}'
FULL_WORK = Path('/results') / f'public-full-{STAMP}'

os.environ.setdefault('COSMOS_API_BASE', 'http://cosmos-endpoint:8080/v1/chat/completions')
# Do not overwrite COSMOS_MODEL: deployment/BYOM configuration selects it.
print(f'Source: {SOURCE}; smoke output: {SMOKE_WORK}; full run enabled: {RUN_FULL}')

### Optional · Bring your own model

Prefer setting `COSMOS_API_BASE`, `COSMOS_MODEL`, and an optional secret `COSMOS_API_KEY` at **initial deployment**: setup then skips the local GPU endpoint. For a session-only override, set `USE_BYOM = True` below **before** the health check. The URL must end in `/v1/chat/completions` and be reachable from the evaluator container. The service must accept the harness's image/video requests. Do not put credentials in the URL.

Changing these settings here does not stop an already-running local GPU container. Leave this cell disabled to preserve all inherited settings. A blank hidden key prompt preserves the inherited key.

In [ ]:
USE_BYOM = False
if USE_BYOM:
    os.environ['COSMOS_API_BASE'] = input('Full chat/completions URL (no embedded credentials): ').strip()
    os.environ['COSMOS_MODEL'] = input('Exact served model ID: ').strip()
    byom_key = getpass('Endpoint key (hidden; blank preserves inherited key): ')
    if byom_key:
        os.environ['COSMOS_API_KEY'] = byom_key
    del byom_key

## 2 · Prepare VQA with the selected harness

Public mode uses the official public preparation path and pinned HF revision. VQA-only preparation downloads the VQA subtree rather than the entire benchmark; it still needs real media and can take time. Internal mode instead uses the authorized internal S3/cache constructor. Public and internal data are isolated under `/data/public/LMUData` and `/data/internal/LMUData`.

In [ ]:
subprocess.run(CLI + ['prepare', '--source', SOURCE, '--benchmarks', 'VANTAGE_VQA'], check=True)

## 3 · Wait for the endpoint

Wait up to 30 minutes for `/v1/models` and verify the requested model is served. This checks readiness, not GPU compatibility or benchmark quality. Failures propagate instead of being suppressed. The local model API is available only on the Compose network; do not expose it publicly.

In [ ]:
subprocess.run(CLI + ['check', '--wait', '1800'], check=True)

## 4 · Infer five real VQA questions

The helper applies a deterministic five-row VQA selection (`limit=5`, `random_state=0`) and runs inference only. It checks actual prediction artifacts because the native harness can catch exceptions and exit zero without producing usable results. This is an integration smoke, **not** a full benchmark, local score, or valid submission.

In [ ]:
subprocess.run(CLI + ['run', '--source', SOURCE, '--benchmarks', 'VANTAGE_VQA',
                      '--mode', 'infer', '--smoke', '--work-dir', str(SMOKE_WORK)], check=True)

## 5 · Inspect smoke results

Inspect prediction paths and run evidence. Native outputs can be TSV rather than XLSX. Public labels are evaluated by the official server; no local public leaderboard score is claimed. Use Jupyter's `/results` browser for logs, generated configs, and provenance; logs may contain dataset/model content, so share carefully.

In [ ]:
subprocess.run(CLI + ['results', '--source', SOURCE, '--work-dir', str(SMOKE_WORK)], check=True)

## 6 · Opt in to the full eight-task public run

**Disabled by default.** Set `RUN_FULL = True` in the configure cell, then run this section. Use `SOURCE = 'public'`. The following cells otherwise do no preparation, inference, or export. Full preparation needs substantial disk/network time: roughly 42 GB public data plus roughly 16 GB SOT source data, in addition to weights, containers, and extraction/cache headroom. SOT requires the official SmartSpaces license/access prerequisites.

All eight tasks run: VQA, EventVerification, 2DGrounding, 2DPointing, Astro2D, Temporal, DVC, and SOT. No runtime or score-parity promise is made. The public harness, dataset classes, prompts, and ID emission differ from the internal fork even where sampling/model parameters are preserved.

This run requests **inference**, then the official submission packager. Public `--mode all` also uses inference because public ground truth is held out; it does not produce a local leaderboard score. Never reuse the five-question directory for this run.

In [ ]:
if RUN_FULL:
    if SOURCE != 'public':
        raise ValueError('This full-submission section requires SOURCE = public.')
    if not os.environ.get('HF_TOKEN'):
        token = getpass('HF read token with accepted SmartSpaces access (hidden): ').strip()
        if not token:
            raise ValueError('Full SOT preparation requires authorized SmartSpaces access and HF_TOKEN.')
        os.environ['HF_TOKEN'] = token
        del token
    subprocess.run(CLI + ['prepare', '--source', 'public', '--benchmarks', 'all'], check=True)
else:
    print('Full public run disabled; set RUN_FULL = True to opt in.')

In [ ]:
if RUN_FULL:
    if SOURCE != 'public':
        raise ValueError('This full-submission section requires SOURCE = public.')
    subprocess.run(CLI + ['run', '--source', 'public', '--benchmarks', 'all',
                          '--mode', 'infer', '--work-dir', str(FULL_WORK)], check=True)

In [ ]:
if RUN_FULL:
    subprocess.run(CLI + ['results', '--source', 'public', '--work-dir', str(FULL_WORK)], check=True)

## 7 · Export and download the official submission

The public helper validates complete task coverage and uses the official packager, not an invented archive schema. Pillars require complete task sets: semantic = VQA + EventVerification; spatial = Grounding + Pointing + Astro2D; temporal = Temporal + DVC; tracking = SOT. This notebook's submission path runs **all eight**, not a partial pillar or smoke.

Download the generated artifact using the authenticated Jupyter link below, then follow [official submission instructions](https://github.com/Clemson-Capstone/VANTAGE-Bench/blob/ec1ebdebb955332a9d7fdd9e98f39627b2822ff5/SUBMISSION.md) and submit at [vantage-bench.org/submit](https://vantage-bench.org/submit). Server scoring may take up to 24 hours. Nothing is uploaded automatically. Clear notebook outputs before sharing; do not redistribute restricted dataset media or credentials.

In [ ]:
if RUN_FULL:
    subprocess.run(CLI + ['export', '--source', 'public', '--work-dir', str(FULL_WORK)], check=True)
    archive = FULL_WORK.parent / f'{FULL_WORK.name}-submission.tar.gz'
    display(FileLink(str(archive), url_prefix='/files'))

## Optional · Authorized internal evaluation

Public VQA smoke supports anonymous HF access. For internal mode, configure **at initial deployment** an existing host `AWS_CREDENTIALS_DIR` containing AWS `config` and `credentials` with the authorized `team-cosmos` profile (or securely populate the default `state/credentials/aws` before setup). Compose mounts it read-only under `/credentials/aws`, using `AWS_CONFIG_FILE=/credentials/aws/config` and `AWS_SHARED_CREDENTIALS_FILE=/credentials/aws/credentials`. Credentials must remain valid during preparation. A notebook variable cannot create or change a host bind mount. Never paste AWS values into this notebook.

Then set `SOURCE = 'internal'` in section 1, keep `RUN_FULL = False`, and rerun configuration, VQA preparation, health check, smoke, and results. Those same commands pass `--source internal` to the NVIDIA harness. Internal data stays under `/data/internal/LMUData`; public data stays under `/data/public/LMUData`, with separate Python/import roots as well. Internal `export --source internal` creates an evidence bundle, **not** a public leaderboard submission. See the README for CLI examples.

These eight VANTAGE tasks need no additional dataset-service, NGC, or API judge keys. TAR and other benchmark families are outside scope.

### Stop/restart safely
Use the **VM terminal**, in the launchable directory: `docker compose --profile local stop` preserves results/caches. For existing local services, `docker compose --profile local start cosmos-endpoint vantage-eval`; for BYOM, `docker compose start vantage-eval`. For changed configuration rerun setup with the intended deployment environment. See README for log commands. Stopping containers does not stop VM billing.